## 1. Imports

In [1]:
import os, json, time, ast
import pandas as pd
import httpx
from dotenv import load_dotenv
load_dotenv()

# ── RAGAS imports ─────────────────────────────────────────────────────────────
from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.metrics import (
    LLMContextPrecisionWithReference,
    LLMContextRecall,
    Faithfulness,
    AnswerRelevancy,
)
from ragas.llms       import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# ── Visualisation ─────────────────────────────────────────────────────────────
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    import numpy as np
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print('matplotlib not installed — charts will be skipped.')
    print('Install with: pip install matplotlib --break-system-packages')

c:\Users\GIGA\OneDrive - Universiti Malaya\Documents\rag-for-beginners\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\GIGA\AppData\Local\Temp\ipykernel_37052\4098928738.py:9: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference
  from ragas.metrics import (
C:\Users\GIGA\AppData\Local\Temp\ipykernel_37052\4098928738.py:9: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import (
C:\Users\GIGA\AppData\Local\Tem

## 2. Judge LLM (for RAGAS scoring)

In [2]:
judge_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
judge_emb = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

C:\Users\GIGA\AppData\Local\Temp\ipykernel_37052\757179373.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
C:\Users\GIGA\AppData\Local\Temp\ipykernel_37052\757179373.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  judge_emb = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


## 3. Load wound-specific testset

In [3]:
OUTPUT_DIR   = "../ragas_testset/"
TESTSET_JSON  = os.path.join(OUTPUT_DIR, 'wound_testset_curated.json')
TESTSET_CSV   = os.path.join(OUTPUT_DIR, 'wound_testset_curated.csv')

if os.path.isfile(TESTSET_JSON):
    with open(TESTSET_JSON, 'r', encoding='utf-8') as f:
        testset = json.load(f)
    print(f'Loaded {len(testset)} test cases from JSON')
elif os.path.isfile(TESTSET_CSV):
    df_ts = pd.read_csv(TESTSET_CSV)
    testset = []
    for _, row in df_ts.iterrows():
        try:
            time_inputs = ast.literal_eval(row['time_inputs']) if isinstance(row['time_inputs'], str) else row['time_inputs']
        except Exception:
            time_inputs = {}
        try:
            ref_ctx = ast.literal_eval(row['reference_contexts']) if isinstance(row['reference_contexts'], str) else []
        except Exception:
            ref_ctx = []
        testset.append({
            'synthesizer_name':   str(row.get('synthesizer_name', '')),
            'time_inputs':        time_inputs,
            'user_input':         str(row.get('user_input', '')),
            'reference':          str(row.get('reference', '')),
            'reference_contexts': ref_ctx,
        })
    print(f'Loaded {len(testset)} test cases from CSV')
else:
    raise FileNotFoundError(f'Testset not found at {TESTSET_JSON} or {TESTSET_CSV}')


Loaded 21 test cases from JSON


## 4. Architecture caller — sends T.I.M.E. fields to the server

**IMPORTANT**: for each experiment, you change ONE thing in wound_app.py,
restart the server, then run this function. The function itself does not change.

In [4]:
RAG_URL = "http://localhost:8000/get_recommendation"

def call_rag(record: dict, timeout: int = 120) -> dict:
    """Send T.I.M.E. inputs to the running server. Uses time_inputs directly."""
    t = record["time_inputs"]
    payload = {
        "necrotic_pct":    t["necrotic_pct"],
        "slough_pct":      t["slough_pct"],
        "granulation_pct": t["granulation_pct"],
        "infection":       t["infection"],
        "moisture":        t["moisture"],
        "edge":            t["edge"],
        "notes":           record["user_input"],   # clinical scenario as notes
        "tissue_confidence": 0.0,
    }
    try:
        r = httpx.post(RAG_URL, data=payload, timeout=timeout)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        return {"result": f"ERROR: {e}", "chunk_texts": [], "confidence_label": "LOW"}

## 5. Evaluation runner — call this once per architecture

In [5]:
METRIC_COLS = [
    'llm_context_precision_with_reference',
    'context_recall',
    'faithfulness',
    'answer_relevancy',
]
METRIC_LABELS = [
    'Context precision',
    'Context recall',
    'Faithfulness',
    'Answer relevancy',
]
METRIC_COLORS = ['#185FA5', '#1D9E75', '#BA7517', '#D85A30']


def run_evaluation(experiment_name: str, results_json: str):
    """
    Calls the RAG server for every test case, scores with RAGAS,
    saves charts into the same folder as results_json, and returns
    (scores_df, agg_dict).
    """
    print(f"\n{'='*60}")
    print(f'EXPERIMENT: {experiment_name}')
    print(f"{'='*60}")

    if os.path.isfile(results_json):
        with open(results_json, encoding='utf-8') as f:
            records = json.load(f)
        done = {r['index'] for r in records}
        print(f'Resuming: {len(records)}/{len(testset)} done')
    else:
        records = []
        done    = set()

    for idx, case in enumerate(testset):
        if idx in done:
            print(f'  [{idx+1:>2}/{len(testset)}] skip (done)')
            continue

        print(f"  [{idx+1:>2}/{len(testset)}] {case['synthesizer_name']}")
        t0      = time.time()
        resp    = call_rag(case)
        elapsed = time.time() - t0

        answer = resp.get('result', '')
        chunks = resp.get('chunk_texts', [])
        retrieved_contexts = chunks if chunks else [answer]

        records.append({
            'index':               idx,
            'synthesizer_name':    case['synthesizer_name'],
            'user_input':          case['user_input'],
            'reference':           case['reference'],
            'reference_contexts':  case['reference_contexts'],
            'retrieved_contexts':  retrieved_contexts,
            'answer':              answer,
            'confidence_label':    resp.get('confidence_label', '?'),
            'elapsed_sec':         round(elapsed, 1),
        })

        with open(results_json, 'w', encoding='utf-8') as f:
            json.dump(records, f, indent=2, ensure_ascii=False)

        time.sleep(1.0)

    # ── Build RAGAS dataset ───────────────────────────────────────────────────
    samples = []
    for r in records:
        if not r['answer'] or r['answer'].startswith('ERROR'):
            continue
        samples.append(SingleTurnSample(
            user_input          = r['user_input'],
            reference           = r['reference'],
            reference_contexts  = [str(c) for c in r['reference_contexts']],
            retrieved_contexts  = [str(c) for c in r['retrieved_contexts']],
            response            = r['answer'],
        ))

    print(f'\nRunning RAGAS scoring on {len(samples)} samples...')
    dataset = EvaluationDataset(samples)
    results = evaluate(dataset, metrics=[
        LLMContextPrecisionWithReference(llm=judge_llm),
        LLMContextRecall(llm=judge_llm),
        Faithfulness(llm=judge_llm),
        AnswerRelevancy(llm=judge_llm, embeddings=judge_emb),
    ])

    scores_df = results.to_pandas()
    metric_cols = [c for c in scores_df.columns
                   if c not in ('user_input', 'reference', 'retrieved_contexts',
                                'reference_contexts', 'response')]

    # ── merge synthesizer names ───────────────────────────────────────────────
    valid_records = [r for r in records if r['answer'] and not r['answer'].startswith('ERROR')]
    full_df = scores_df.copy()
    full_df['synthesizer_name'] = [r['synthesizer_name'] for r in valid_records[:len(full_df)]]
    full_df['confidence_label'] = [r['confidence_label']  for r in valid_records[:len(full_df)]]
    full_df['elapsed_sec']      = [r['elapsed_sec']        for r in valid_records[:len(full_df)]]

    print(f'\nResults for {experiment_name}:')
    agg = {'experiment': experiment_name}
    for col in metric_cols:
        mean = scores_df[col].mean()
        agg[col] = round(mean, 4)
        label = METRIC_LABELS[METRIC_COLS.index(col)] if col in METRIC_COLS else col
        print(f'  {label:<35} {mean:.4f} ({mean:.1%})')

    # ── Save charts ───────────────────────────────────────────────────────────
    if HAS_MPL:
        _save_charts(experiment_name, results_json, full_df, metric_cols, agg)

    return pd.DataFrame([agg]), full_df


# ─────────────────────────────────────────────────────────────────────────────
# CHART GENERATION
# ─────────────────────────────────────────────────────────────────────────────

def _save_charts(experiment_name, results_json, full_df, metric_cols, agg):
    """Generate and save 3 charts into the experiment folder."""
    plots_dir = os.path.join(os.path.dirname(results_json), 'plots')
    os.makedirs(plots_dir, exist_ok=True)

    # colour map — use METRIC_COLORS when column order matches, else cycle
    colors = [METRIC_COLORS[METRIC_COLS.index(c)] if c in METRIC_COLS else '#888780'
              for c in metric_cols]
    labels = [METRIC_LABELS[METRIC_COLS.index(c)] if c in METRIC_COLS else c
              for c in metric_cols]

    # ── Plot 1: Aggregate horizontal bar chart ────────────────────────────────
    fig, ax = plt.subplots(figsize=(9, 4))
    means = [agg.get(c, 0) for c in metric_cols]
    bars  = ax.barh(labels, means, color=colors, height=0.5, zorder=2)
    ax.set_xlim(0, 1.0)
    ax.axvline(0.70, color='#f59e0b', linestyle='--', linewidth=1.2, label='0.70 threshold')
    ax.axvline(0.85, color='#22c55e', linestyle='--', linewidth=1.2, label='0.85 excellent')
    for bar, val in zip(bars, means):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.3f} ({val:.0%})', va='center', fontsize=10)
    ax.set_title(f'VerdaSense RAG — {experiment_name}\nRAGAS Aggregate Scores',
                 fontsize=12, fontweight='bold', pad=10)
    ax.set_xlabel('Score (0 = worst, 1 = best)')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(axis='x', alpha=0.3, zorder=1)
    ax.set_facecolor('#f8fafc')
    fig.tight_layout()
    p1 = os.path.join(plots_dir, 'aggregate_scores.png')
    fig.savefig(p1, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  ✅  Chart 1 saved: {p1}')

    # ── Plot 2: Per-question heatmap ──────────────────────────────────────────
    score_matrix = full_df[metric_cols].values
    n_rows = len(full_df)
    fig, ax = plt.subplots(figsize=(11, max(5, n_rows * 0.42 + 1.5)))
    im = ax.imshow(score_matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
    ax.set_xticks(range(len(metric_cols)))
    ax.set_xticklabels([l.replace(' ', '\n') for l in labels], fontsize=9)
    ax.set_yticks(range(n_rows))
    y_labels = [f"Q{i+1} {full_df.iloc[i]['synthesizer_name'][:18]}"
                for i in range(n_rows)]
    ax.set_yticklabels(y_labels, fontsize=8)
    for i in range(score_matrix.shape[0]):
        for j in range(score_matrix.shape[1]):
            val = score_matrix[i, j]
            if val != val:   # NaN guard
                continue
            txt_col = 'black' if 0.25 < val < 0.75 else 'white'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=8, color=txt_col)
    plt.colorbar(im, ax=ax, label='Score')
    ax.set_title(f'{experiment_name} — Per-Question RAGAS Heatmap',
                 fontsize=11, fontweight='bold', pad=10)
    fig.tight_layout()
    p2 = os.path.join(plots_dir, 'per_question_heatmap.png')
    fig.savefig(p2, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  ✅  Chart 2 saved: {p2}')

    # ── Plot 3: Grouped bars by synthesizer type ──────────────────────────────
    synth_groups = full_df.groupby('synthesizer_name')[metric_cols].mean()
    x  = np.arange(len(synth_groups))
    w  = 0.18
    n  = len(metric_cols)
    fig, ax = plt.subplots(figsize=(max(10, len(synth_groups)*0.9), 5))
    for j, (col, color, label) in enumerate(zip(metric_cols, colors, labels)):
        vals = synth_groups[col].values
        ax.bar(x + j*w - w*(n-1)/2, vals, width=w, label=label,
               color=color, alpha=0.88)
    ax.set_xticks(x)
    ax.set_xticklabels(synth_groups.index, fontsize=7, rotation=40, ha='right')
    ax.set_ylim(0, 1.0)
    ax.axhline(0.70, color='#f59e0b', linestyle='--', linewidth=1, alpha=0.8)
    ax.set_title(f'{experiment_name} — RAGAS Scores by Test Case Type',
                 fontsize=11, fontweight='bold')
    ax.set_ylabel('Score')
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(axis='y', alpha=0.3)
    fig.tight_layout()
    p3 = os.path.join(plots_dir, 'by_synthesizer.png')
    fig.savefig(p3, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  ✅  Chart 3 saved: {p3}')

    # ── Plot 4: Radar chart ────────────────────────────────────────────────────
    _radar_chart(experiment_name, metric_cols, labels, colors, agg, plots_dir)


def _radar_chart(experiment_name, metric_cols, labels, colors, agg, plots_dir):
    """Radar / spider chart for aggregate scores."""
    vals   = [agg.get(c, 0) for c in metric_cols]
    N      = len(metric_cols)
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]
    vals_r  = vals + vals[:1]

    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
    ax.plot(angles, vals_r, 'o-', linewidth=2, color='#185FA5')
    ax.fill(angles, vals_r, alpha=0.20, color='#185FA5')
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.70, 0.85, 1.0])
    ax.set_yticklabels(['0.25', '0.50', '0.70', '0.85', '1.0'], fontsize=7)
    ax.set_title(f'{experiment_name}\nRAGAS Radar', fontsize=11,
                 fontweight='bold', pad=20)
    # annotate values
    for angle, val, label in zip(angles[:-1], vals, labels):
        ax.annotate(f'{val:.2f}',
                    xy=(angle, val),
                    xytext=(angle, val + 0.07),
                    fontsize=9, ha='center', color='#0C447C')
    fig.tight_layout()
    p4 = os.path.join(plots_dir, 'radar_chart.png')
    fig.savefig(p4, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  ✅  Chart 4 saved: {p4}')


In [25]:
# Change these two strings for each new experiment:
#   experiment_name  → label used in charts and printed output
#   results_json     → path inside the experiment folder
#
# Then restart the server pointing at the matching wound_app_XX.py, and run.

agg_df_01, full_df_01 = run_evaluation(
    experiment_name = 'wound_ragas_eval_01',
    results_json    = '../ragas_eval_01/wound_ragas_ablation_results.json',
)
agg_df_01



EXPERIMENT: wound_ragas_eval_01
  [ 1/21] type1_granulating_dry
  [ 2/21] type1_granulating_wet
  [ 3/21] type2_clean_wet
  [ 4/21] type3_dry_infected_low_necrotic
  [ 5/21] type3_iodine_contraindication
  [ 6/21] type4_wet_infected_low_necrotic
  [ 7/21] type5_dry_noninfected_high_necrotic
  [ 8/21] type6_wet_noninfected_high_necrotic
  [ 9/21] type7_dry_infected_high_necrotic
  [10/21] type8_wet_infected_high_necrotic
  [11/21] notes_infection_override
  [12/21] diabetic_foot_wound
  [13/21] diabetic_high_risk_nonhealing
  [14/21] skin_tear_elderly
  [15/21] postoperative_clean
  [16/21] burn_hand_referral
  [17/21] dressing_change_saturation
  [18/21] npwt_contraindication
  [19/21] silver_contraindicated_granulating
  [20/21] time_assessment_mixed_wound
  [21/21] dressing_selection_heavy_exudate

Running RAGAS scoring on 11 samples...


Evaluating: 100%|██████████| 44/44 [04:13<00:00,  5.76s/it]



Results for wound_ragas_eval_01:
  Context precision                   1.0000 (100.0%)
  Context recall                      0.8553 (85.5%)
  Faithfulness                        0.7316 (73.2%)
  Answer relevancy                    0.5775 (57.7%)
  ✅  Chart 1 saved: ../ragas_eval_01\plots\aggregate_scores.png
  ✅  Chart 2 saved: ../ragas_eval_01\plots\per_question_heatmap.png
  ✅  Chart 3 saved: ../ragas_eval_01\plots\by_synthesizer.png
  ✅  Chart 4 saved: ../ragas_eval_01\plots\radar_chart.png


,experiment,llm_context_precision_with_reference,context_recall,faithfulness,answer_relevancy
0,wound_ragas_eval_01,1.0,0.8553,0.7316,0.5775


In [6]:
agg_df_02, full_df_02 = run_evaluation(
    experiment_name = 'wound_ragas_eval_02',
    results_json    = '../ragas_eval_02/wound_ragas_ablation_results.json',
)
agg_df_02


EXPERIMENT: wound_ragas_eval_02
  [ 1/21] type1_granulating_dry
  [ 2/21] type1_granulating_wet
  [ 3/21] type2_clean_wet
  [ 4/21] type3_dry_infected_low_necrotic
  [ 5/21] type3_iodine_contraindication
  [ 6/21] type4_wet_infected_low_necrotic
  [ 7/21] type5_dry_noninfected_high_necrotic
  [ 8/21] type6_wet_noninfected_high_necrotic
  [ 9/21] type7_dry_infected_high_necrotic
  [10/21] type8_wet_infected_high_necrotic
  [11/21] notes_infection_override
  [12/21] diabetic_foot_wound
  [13/21] diabetic_high_risk_nonhealing
  [14/21] skin_tear_elderly
  [15/21] postoperative_clean
  [16/21] burn_hand_referral
  [17/21] dressing_change_saturation
  [18/21] npwt_contraindication
  [19/21] silver_contraindicated_granulating
  [20/21] time_assessment_mixed_wound
  [21/21] dressing_selection_heavy_exudate

Running RAGAS scoring on 21 samples...


Evaluating: 100%|██████████| 84/84 [06:13<00:00,  4.44s/it]



Results for wound_ragas_eval_02:
  Context precision                   1.0000 (100.0%)
  Context recall                      0.7002 (70.0%)
  Faithfulness                        0.7798 (78.0%)
  Answer relevancy                    0.6059 (60.6%)
  ✅  Chart 1 saved: ../ragas_eval_02\plots\aggregate_scores.png
  ✅  Chart 2 saved: ../ragas_eval_02\plots\per_question_heatmap.png
  ✅  Chart 3 saved: ../ragas_eval_02\plots\by_synthesizer.png
  ✅  Chart 4 saved: ../ragas_eval_02\plots\radar_chart.png


,experiment,llm_context_precision_with_reference,context_recall,faithfulness,answer_relevancy
0,wound_ragas_eval_02,1.0,0.7002,0.7798,0.6059
